# Projeto: Modelo Preditivo de Risco de Crédito
## Desenvolvimento de ferramenta para suporte à decisão de concessão de crédito.  
### Objetivo: Identificar o perfil de clientes com maior probabilidade de inadimplência utilizando o dataset histórico.

## Importação das bibliotecas

In [ ]:

import pandas as pd
import sklearn
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None) #Para mostrar todas as colunas
pd.set_option('display.float_format', '{:.2f}'.format) #Para formatar decimais

## Importando df e renomeando as colunas

In [ ]:
df = pd.read_csv('credit_risk_dataset.csv')
dicionario_nomes_colunas = {
    'person_age': 'idade',
    'person_income': 'renda_anual',
    'person_home_ownership': 'tipo_residencia',
    'person_emp_length': 'tempo_trabalho',
    'loan_intent': 'motivo_emprestimo',
    'loan_grade': 'nota_credito',
    'loan_amnt': 'valor_emprestimo',
    'loan_int_rate': 'taxa_juros',
    'loan_status': 'inadimplente',
    'loan_percent_income': 'percentual_comprometimento',
    'cb_person_default_on_file': 'historico_inadimplencia',
    'cb_person_cred_hist_length': 'tempo_historico_credito'
}
df = df.rename(columns=dicionario_nomes_colunas)
display(df.head())

Verificação de Integridade dos Dados (Valores Ausentes)

In [ ]:
df.isnull().sum() #Notado que a coluna 'taxa_juros'e 'tempo_trabalho' possuem valores nulos, que tratarei na próxima etapa

Tratamento de Dados Ausentes
Identifiquei valores nulos nas variáveis taxa_juros e tempo_trabalho. Para manter a integridade da base e permitir o treinamento do modelo, realizarei a imputação por mediana.
(A tentativa por mediana se deve por ser uma medida de tendência rovusta a outliers garantindo que valores extremos não deixem o modelo enviesado).

In [ ]:
mediana_juros = df['taxa_juros'].median()
mediana_tempo = df['tempo_trabalho'].median()

df['taxa_juros'] = df['taxa_juros'].fillna(mediana_juros)
df['tempo_trabalho'] = df['tempo_trabalho'].fillna(mediana_tempo)

print(df[['taxa_juros', 'tempo_trabalho']].isnull().sum())

Identificando inconsistências através da estatística descritiva.

In [35]:
df.describe()

,idade,renda_anual,tempo_trabalho,valor_emprestimo,taxa_juros,inadimplente,percentual_comprometimento,tempo_historico_credito
count,32581.000000,3.258100e+04,32581.000000,32581.000000,32581.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.767994,9589.371106,11.009620,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.087372,6322.086646,3.081611,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,8.490000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.110000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


Encontrado inconsistência na idade max e tempo de trabalho, aplicarei filtros para remover os registros.

In [37]:
df = df[ df['idade'] < 100]
df = df[ df['tempo_trabalho'] < 60]

df.describe()

,idade,renda_anual,tempo_trabalho,valor_emprestimo,taxa_juros,inadimplente,percentual_comprometimento,tempo_historico_credito
count,32574.000000,3.257400e+04,32574.000000,32574.000000,32574.000000,32574.000000,32574.000000,32574.000000
mean,27.718426,6.587848e+04,4.760576,9588.018051,11.009470,0.218180,0.170202,5.804108
std,6.204987,5.253194e+04,3.981181,6320.249598,3.081664,0.413017,0.106755,4.053873
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,8.490000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.110000,0.000000,0.230000,8.000000
max,94.000000,2.039784e+06,41.000000,35000.000000,23.220000,1.000000,0.830000,30.000000
